# Assignment 4: Nonlinear Forecasting and Risk Control
**ECON 417 Business Forecasting — Southern Illinois University Edwardsville**

**Student Version**

---
### Instructions
This assignment covers Lecture 4: Nonlinear Forecasting and Risk Control. Topics include polynomial and interaction terms, decision trees, tree depth tuning, random forests, and feature importance.

- **Q4 and Q7** are coding questions — complete the code cells.
- All other questions are written responses — type your answers in the answer cells.
- The extended feature set from Q4 is reused in Q7.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# ── Load data (yfinance with synthetic fallback) ───────────────────────────
try:
    import yfinance as yf
    _raw = yf.download(['^GSPC','^VIX','MSFT','AAPL','JPM','XOM'],
                       start='2015-01-01', end='2024-12-31', auto_adjust=True)
    _close = _raw['Close']
    df = pd.DataFrame({
        'sp500':    _close['^GSPC'].pct_change(),
        'vix_chg':  _close['^VIX'].diff(),
        'lag_msft': _close['MSFT'].pct_change().shift(1),
        'lag_aapl': _close['AAPL'].pct_change().shift(1),
        'lag_jpm':  _close['JPM'].pct_change().shift(1),
        'lag_xom':  _close['XOM'].pct_change().shift(1),
    }).dropna()
    print("✓ Live data loaded from yfinance.")
except Exception:
    print("yfinance unavailable – using synthetic data.")
    np.random.seed(42)
    dates = pd.date_range('2015-01-02', '2024-12-31', freq='B')
    n = len(dates)
    sp500 = np.random.randn(n) * 0.01
    msft  = sp500 * 1.10 + np.random.randn(n) * 0.008
    aapl  = sp500 * 1.05 + np.random.randn(n) * 0.009
    jpm   = sp500 * 0.90 + np.random.randn(n) * 0.010
    xom   = sp500 * 0.70 + np.random.randn(n) * 0.010
    vix   = -sp500 * 100 + np.random.randn(n) * 2
    df = pd.DataFrame({
        'sp500':    sp500,
        'vix_chg':  np.concatenate([[np.nan], np.diff(vix)]),
        'lag_msft': np.concatenate([[np.nan], msft[:-1]]),
        'lag_aapl': np.concatenate([[np.nan], aapl[:-1]]),
        'lag_jpm':  np.concatenate([[np.nan], jpm[:-1]]),
        'lag_xom':  np.concatenate([[np.nan], xom[:-1]]),
    }, index=dates).dropna()

# ── 60 / 20 / 20 chronological split ─────────────────────────────────────
n  = len(df)
t1 = int(n * 0.60)
t2 = int(n * 0.80)

df_train = df.iloc[:t1]
df_val   = df.iloc[t1:t2]
df_test  = df.iloc[t2:]

base_features = ['vix_chg', 'lag_msft', 'lag_aapl', 'lag_jpm', 'lag_xom']
target = 'sp500'

# ── Baseline linear model ─────────────────────────────────────────────────
X_tr_base = df_train[base_features].values
X_va_base = df_val[base_features].values
X_te_base = df_test[base_features].values
y_train   = df_train[target].values
y_val     = df_val[target].values
y_test    = df_test[target].values

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

ols_base = LinearRegression().fit(X_tr_base, y_train)
print(f"Baseline OLS  — Train RMSE: {rmse(y_train, ols_base.predict(X_tr_base)):.6f} "
      f"| Val RMSE: {rmse(y_val, ols_base.predict(X_va_base)):.6f}")
print(f"Data: {len(df_train)} train / {len(df_val)} val / {len(df_test)} test observations")


## Q1. Nonlinearity in Forecasting (Written)

What does **nonlinearity** mean in the context of a forecasting model? Give a concrete financial markets example — you may use VIX as a reference.

**Your Answer:**

_Type your answer here._

## Q2. Polynomial Terms (Written)

Explain how polynomial terms extend a linear regression model. If we add ΔVIX² to our model, does the model become **nonlinear**? In what sense is it still 'linear'?

**Your Answer:**

_Type your answer here._

## Q3. Interaction Terms (Written)

What is an **interaction term**? In the model:

$$r_t^{SP500} = \beta_0 + \beta_1 \Delta VIX_t + \beta_2 \Delta VIX_t^2 + \beta_3 (\mathbf{1}_{HighVIX,t} \times r_{t-1}^{MSFT}) + u_t$$

What does β₃ represent economically?

**Your Answer:**

_Type your answer here._

## Q4. ⭐ Coding: Nonlinear Feature Engineering

Build an extended feature matrix by adding to `base_features`:
- `vix_chg_sq` = ΔVIX²
- `high_vix` = 1 if `vix_chg` > 75th-percentile of training data, else 0
- `hv_x_msft` = `high_vix` × `lag_msft`

Fit OLS on the extended features. Compare Train/Val RMSE to the baseline linear model. Print the coefficient for `hv_x_msft` and interpret it.

In [ ]:
# ── Q4: Nonlinear / Interaction Feature Engineering ─────────────────────
# TODO: Build an extended feature matrix with the following additions:
#   (a) vix_chg_sq  = vix_chg squared
#   (b) high_vix    = 1 if vix_chg > 75th percentile of df_train['vix_chg'], else 0
#   (c) hv_x_msft   = high_vix × lag_msft  (interaction term)
#
# Fit OLS on the extended training features and compare Train/Val RMSE
# to the baseline linear model.

ext_features = base_features + ['vix_chg_sq', 'high_vix', 'hv_x_msft']

# YOUR CODE HERE: create X_tr_ext, X_va_ext, X_te_ext using the new features
# YOUR CODE HERE: fit ols_ext = LinearRegression() on extended training data
# YOUR CODE HERE: print Val RMSE comparison between baseline and extended model

print("Baseline OLS  — Val RMSE:", rmse(y_val, ols_base.predict(X_va_base)))
# YOUR CODE HERE: print Extended OLS Val RMSE


## Q5. Decision Trees (Written)

Explain how a **decision tree** makes predictions for a regression task. How does it partition the feature space differently from a linear regression?

**Your Answer:**

_Type your answer here._

## Q6. Tree Depth and Overfitting (Written)

What is the role of **`max_depth`** in a decision tree? Explain the connection to overfitting.

**Your Answer:**

_Type your answer here._

## Q7. ⭐ Coding: Tree Depth Search + Random Forest

Use the **extended feature set** (`X_tr_ext`, `X_va_ext`, `X_te_ext`) from Q4.

**(a)** Fit `DecisionTreeRegressor` for `max_depth` ∈ {1, …, 12}. Plot train RMSE and val RMSE vs depth. Mark the best depth.

**(b)** Fit `RandomForestRegressor(n_estimators=200, random_state=42)`. Print a comparison table of Val/Test RMSE for: Extended OLS, Best Tree, Random Forest.

**(c)** Plot a bar chart of `rf.feature_importances_` sorted from highest to lowest.

In [ ]:
# ── Q7a: Decision Tree Depth Search ──────────────────────────────────────
# TODO: Fit DecisionTreeRegressor for max_depth in range(1, 13).
# Record train and val RMSE for each depth using the extended feature set (X_tr_ext, X_va_ext).
# Plot train and val RMSE vs depth. Mark the best depth.

depths = list(range(1, 13))
train_rmse_tree = []
val_rmse_tree   = []

for d in depths:
    # YOUR CODE HERE
    pass

if not val_rmse_tree:
    print("⚠️  Complete the depth loop above, then re-run.")
else:
    best_depth = depths[np.argmin(val_rmse_tree)]
    print(f"Best depth: {best_depth}")
    # YOUR CODE HERE: plot train/val RMSE curves vs depth

# ── Q7b: Random Forest ───────────────────────────────────────────────────
# TODO: Fit a RandomForestRegressor with n_estimators=200, random_state=42.
# Print a comparison table of Val/Test RMSE for:
#   - Extended OLS, Best Decision Tree, Random Forest

# YOUR CODE HERE

# ── Q7c: Feature Importance Bar Chart ─────────────────────────────────────
# TODO: Extract rf.feature_importances_, sort descending, and plot a bar chart.
# YOUR CODE HERE


## Q8. Random Forest vs. Single Tree (Written)

Why does a random forest typically **outperform a single decision tree** on out-of-sample data? Explain in terms of bias and variance.

**Your Answer:**

_Type your answer here._

## Q9. Interpreting Feature Importance (Written)

After computing feature importances from a random forest, suppose `lag_msft` scores 0.35 and `lag_xom` scores 0.02. Does this mean lag_msft **causes** S&P 500 returns to move, while lag_xom is **irrelevant**? What are the limitations of feature importance?

**Your Answer:**

_Type your answer here._

## Q10. Flexibility Discipline (Written)

Describe the **'flexibility discipline'** principle from Lecture 4. Why should we start with a linear model and add complexity only when there is both statistical and economic justification?

**Your Answer:**

_Type your answer here._

---
**End of Assignment 4**
*ECON 417 Business Forecasting · Southern Illinois University Edwardsville*